In [1]:
import os
import sys

# 현재 작업 디렉토리 확인해보고
print("CWD:", os.getcwd())

# notebooks/ 에서 한 단계 위(프로젝트 루트)로 올라가기
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
print("PROJECT_ROOT:", PROJECT_ROOT)

# sys.path 에 프로젝트 루트가 없으면 추가
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
    print("Added to sys.path")


CWD: c:\Users\user\Desktop\project\scalp-vision-agent\notebooks
PROJECT_ROOT: c:\Users\user\Desktop\project\scalp-vision-agent
Added to sys.path


In [2]:
import torch
import torch.nn.functional as F

from src.cnn.losses import multihead_ce_loss
from src.cnn.utils import labels_dict_to_tensor
from src.cnn.train import train_one_epoch, get_device,evaluate_one_epoch, train_model
from src.cnn.models import MultiHeadResNet18


import torch
from torch.utils.data import DataLoader
from torch import optim

from src.config import MASTER_INDEX_CSV
from src.cnn.dataset import ScalpDataset

In [3]:
import torchvision.transforms as T

train_transform = T.Compose([
    T.RandomResizedCrop(224, scale=(0.8, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05,
    ),
    T.ToTensor(),
])

val_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])


In [4]:
from torch.utils.data import Subset

train_dataset = ScalpDataset(index_csv=MASTER_INDEX_CSV, split="train", transforms=train_transform)
val_dataset   = ScalpDataset(index_csv=MASTER_INDEX_CSV, split="val",   transforms=val_transform)

print(len(train_dataset), len(val_dataset))


67588 23568


In [5]:
BATCH_SIZE = 32
NUM_EPOCHS = 10
LR = 1e-4


In [6]:
USE_SUBSET = False   # overfit 테스트: True, 전체 학습: False

if USE_SUBSET:
    subset_indices = list(range(100))  
    train_dataset_small = Subset(train_dataset, subset_indices)
    train_loader = DataLoader(train_dataset_small, batch_size=BATCH_SIZE, shuffle=True)
else:
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)


In [7]:
def compute_head_metrics(
    logits: torch.Tensor,  # [B, 6, 4]
    targets: torch.Tensor, # [B, 6]
):
    """
    배치 하나에 대해 head별 CE loss와 accuracy 계산.
    return: (per_head_losses, per_head_accs), 각 길이 6 리스트.
    """
    B, H, C = logits.shape
    per_head_losses = []
    per_head_accs = []

    for h in range(H):
        head_logits = logits[:, h, :]   # [B, 4]
        head_targets = targets[:, h]    # [B]

        # head별 CE loss
        loss_h = F.cross_entropy(head_logits, head_targets, reduction="mean")
        per_head_losses.append(loss_h.item())

        # head별 accuracy
        preds_h = head_logits.argmax(dim=1)      # [B]
        acc_h = (preds_h == head_targets).float().mean().item()
        per_head_accs.append(acc_h)

    return per_head_losses, per_head_accs


In [8]:
@torch.no_grad()
def evaluate_one_epoch_with_heads(model, dataloader, device):
    """
    전체 val 셋 기준:
    - overall loss / acc
    - head별 평균 loss / acc
    를 함께 계산.
    """
    model.eval()

    running_loss = 0.0
    running_correct = 0
    running_total = 0

    # head별 합계(배치 평균의 합)와 배치 수
    sum_loss_heads = torch.zeros(6)
    sum_acc_heads = torch.zeros(6)
    num_batches = 0

    for batch in dataloader:
        images = batch["image"].to(device)
        targets = labels_dict_to_tensor(batch["labels"]).to(device)

        logits = model(images)  # [B, 6, 4]
        loss = multihead_ce_loss(logits, targets)

        B = images.size(0)
        running_loss += loss.item() * B

        preds = logits.argmax(dim=2)
        running_correct += (preds == targets).sum().item()
        running_total += targets.numel()

        # --- head별 metric ---
        loss_list, acc_list = compute_head_metrics(logits, targets)
        sum_loss_heads += torch.tensor(loss_list)
        sum_acc_heads += torch.tensor(acc_list)
        num_batches += 1

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = running_correct / running_total

    mean_loss_heads = (sum_loss_heads / num_batches).tolist()
    mean_acc_heads = (sum_acc_heads / num_batches).tolist()

    return epoch_loss, epoch_acc, mean_loss_heads, mean_acc_heads


In [9]:
device = get_device()
print("device:", device)

model = MultiHeadResNet18().to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4,
)

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
    "val_head_loss": [],  # [[v1, v2, ..., v6], ...]
    "val_head_acc": [],   # [[v1, v2, ..., v6], ...]
}
best_val_loss = float("inf")
best_model_path = "models/multihead_resnet18_headmetrics_best.pth"


device: cuda


In [10]:
HEAD_NAMES = ["value_1", "value_2", "value_3", "value_4", "value_5", "value_6"]

num_epochs = 3

for epoch in range(1, num_epochs + 1):
    print(f"\n===== Epoch {epoch} / {num_epochs} =====")

    train_loss, train_acc = train_one_epoch(
        model=model,
        dataloader=train_loader,
        optimizer=optimizer,
        device=device,
    )

    val_loss, val_acc, val_head_losses, val_head_accs = evaluate_one_epoch_with_heads(
        model=model,
        dataloader=val_loader,
        device=device,
    )

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_head_loss"].append(val_head_losses)
    history["val_head_acc"].append(val_head_accs)

    # best 모델 저장 (val_loss 기준)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)
        print(f"  ↳ Best model updated! val_loss={val_loss:.4f}")

    print(
        f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}"
    )

    # head별 출력
    print("  [Val per-head metrics]")
    for name, h_loss, h_acc in zip(HEAD_NAMES, val_head_losses, val_head_accs):
        print(f"    - {name}: loss={h_loss:.4f}, acc={h_acc:.4f}")



===== Epoch 1 / 3 =====
  ↳ Best model updated! val_loss=0.6454
train_loss=0.6576, train_acc=0.7329 | val_loss=0.6454, val_acc=0.7385
  [Val per-head metrics]
    - value_1: loss=0.4976, acc=0.8465
    - value_2: loss=1.0802, acc=0.5086
    - value_3: loss=0.7585, acc=0.6719
    - value_4: loss=0.2096, acc=0.9516
    - value_5: loss=0.7188, acc=0.7069
    - value_6: loss=0.6059, acc=0.7464

===== Epoch 2 / 3 =====
  ↳ Best model updated! val_loss=0.5885
train_loss=0.6111, train_acc=0.7514 | val_loss=0.5885, val_acc=0.7570
  [Val per-head metrics]
    - value_1: loss=0.4425, acc=0.8532
    - value_2: loss=0.9604, acc=0.5675
    - value_3: loss=0.7196, acc=0.6908
    - value_4: loss=0.1958, acc=0.9526
    - value_5: loss=0.6419, acc=0.7252
    - value_6: loss=0.5691, acc=0.7533

===== Epoch 3 / 3 =====
train_loss=0.5954, train_acc=0.7570 | val_loss=0.5927, val_acc=0.7562
  [Val per-head metrics]
    - value_1: loss=0.4391, acc=0.8563
    - value_2: loss=0.9513, acc=0.5712
    - value_3: